# Systematic bias from three-mass multiplier injections

This notebook recovers the multiplier-based injections generated for

$$
M\in\{10^5,10^6,10^7\}\,M_\odot,
$$

using `SEOBNRv5HMROM` with one active FTI phase parameter at a time. For every mass and injection ID, the GR recovery waveform is aligned once over $\Delta t$, $\phi$, and $\psi$. The same aligned residual is then reused throughout the FTI scan.

On the injection frequency grid, the Fisher matrix and Cutler–Vallisneri source vector are calculated as

$$
\Gamma_{ij}=\left\langle \partial_i h,\partial_j h\right\rangle,
\qquad
\beta_i=\left\langle \partial_i h,\delta h\right\rangle,
$$

with the systematic displacement obtained from $\Delta\theta=-\Gamma^{-1}\beta$. Results are stored separately for each mass, injection ID, and active FTI parameter.


## 0. Imports

In [1]:
import copy
import json
import math
import os

import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from tqdm import tqdm

# Using MathText so that the plots do not require an external LaTeX installation
mpl.rcParams["text.usetex"] = False
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["mathtext.fontset"] = "dejavusans"


In [2]:
import lisabeta.pyconstants as pyconstants
import lisabeta.tools.pytools as pytools
import lisabeta.lisa.pyresponse as pyresponse
import lisabeta.lisa.pyLISAnoise as pyLISAnoise
import lisabeta.lisa.lisatools as lisatools
import lisabeta.lisa.lisa as lisa
import lisabeta.lisa.lisa_fisher as lisa_fisher
import lisabeta.utils.plotutils as plotutils
import lisabeta.inference.inference as inference


/pbs/home/a/amahdi/lisabeta/src/lisabeta/waveforms/bbh/lalsim_wrap.py:10: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [3]:
%matplotlib inline

## 1. Plot labels and numerical utilities

Keeping the parameter names used internally by `lisabeta` unchanged while displaying mathematical labels (using latex) in the figures.


In [4]:
# Defining MathText labels without changing the parameter keys used in calculations

PARAM_LABELS_LATEX = {
    # Labeling the standard GR and source parameters
    "Mchirp": r"$\mathcal{M}$",
    "q": r"$q$",
    "chiPN": r"$\chi_{\rm PN}$",
    "chim": r"$\chi_{-}$",
    "Deltat": r"$\Delta t$",
    "dist": r"$d_L$",
    "inc": r"$\iota$",
    "phi": r"$\varphi_c$",
    "lambda": r"$\lambda$",
    "beta": r"$\beta$",
    "psi": r"$\psi$",

    # Displaying the internal FTI names as deviations of PN phase coefficients
    "fti_dchiMinus2v": r"$\delta\hat{\varphi}_{-2}$",
    "fti_dchi0v": r"$\delta\hat{\varphi}_{0}$",
    "fti_dchi1v": r"$\delta\hat{\varphi}_{1}$",
    "fti_dchi2v": r"$\delta\hat{\varphi}_{2}$",
    "fti_dchi3v": r"$\delta\hat{\varphi}_{3}$",
    "fti_dchi4v": r"$\delta\hat{\varphi}_{4}$",
    "fti_dchi5lv": r"$\delta\hat{\varphi}_{5\ell}$",
    "fti_dchi6v": r"$\delta\hat{\varphi}_{6}$",
    "fti_dchi6lv": r"$\delta\hat{\varphi}_{6\ell}$",
    "fti_dchi7v": r"$\delta\hat{\varphi}_{7}$",
    "fti_dkappa_S": r"$\delta\hat{\kappa}_{S}$",
    "fti_dxi0v": r"$\delta\hat{\xi}_{0}$",
}

CHANNEL_LABELS_LATEX = {
    "TDIA": r"$A$",
    "TDIE": r"$E$",
    "TDIT": r"$T$",
}


def latex_escape_text(text):
    """Escaping underscores for display inside MathText text blocks."""
    return str(text).replace("_", r"\_")


def latex_param_label(param):
    """Returning a MathText label and using roman text for unknown names."""
    return PARAM_LABELS_LATEX.get(param, rf"$\mathrm{{{latex_escape_text(param)}}}$")


def latex_param_labels(params):
    """Returning MathText labels for a sequence of parameter names."""
    return [latex_param_label(p) for p in params]


def latex_param_value(param, value):
    """Formatting a parameter label together with its numerical value."""
    label = latex_param_label(param)
    try:
        value_str = f"{float(value):g}"
    except (TypeError, ValueError):
        value_str = str(value)

    if label.startswith("$") and label.endswith("$"):
        return label[:-1] + rf"={value_str}$"

    return f"{label}={value_str}"


def latex_domega_dtau_label(d_omega=None, d_tau=None):
    """Formatting the injected frequency and damping-time deviations."""
    if d_omega is not None and d_tau is not None:
        try:
            d_omega_str = f"{float(d_omega):g}"
            d_tau_str = f"{float(d_tau):g}"
        except (TypeError, ValueError):
            d_omega_str = str(d_omega)
            d_tau_str = str(d_tau)

        if d_omega_str == d_tau_str:
            return rf"$\delta\Omega=\delta\tau={d_omega_str}$"
        return rf"$\delta\Omega={d_omega_str}$, $\delta\tau={d_tau_str}$"

    if d_omega is not None:
        try:
            d_omega_str = f"{float(d_omega):g}"
        except (TypeError, ValueError):
            d_omega_str = str(d_omega)
        return rf"$\delta\Omega={d_omega_str}$"

    if d_tau is not None:
        try:
            d_tau_str = f"{float(d_tau):g}"
        except (TypeError, ValueError):
            d_tau_str = str(d_tau)
        return rf"$\delta\tau={d_tau_str}$"

    return None


def latex_channel_label(chan):
    """Returning a compact MathText label for a TDI channel."""
    return CHANNEL_LABELS_LATEX.get(chan, rf"$\mathrm{{{latex_escape_text(chan)}}}$")


### Inner product and Fisher-bias calculation

Using the one-sided LISA noise PSD and the injection frequency grid consistently in the overlap, Fisher matrix, and systematic-bias integrals.


In [5]:
def overlap_AET_data(data1, data2, psd):
    """Calculating the summed TDI inner product on a nonuniform frequency grid."""
    freq = data1["freq"]
    diff_freq = np.diff(freq)
    tdi_keys = [key for key in data1 if key != "freq"]
    overlap = 0.0

    # Using the trapezoidal rule for every TDI channel contained in the data
    for channel in tdi_keys:
        integrand = (
            4.0
            * np.real(data1[channel] * np.conj(data2[channel]))
            / psd[channel]
        )
        overlap += np.sum(
            diff_freq * (integrand[1:] + integrand[:-1]) / 2.0
        )

    return overlap


In [6]:
def fisher_beta_data_grid_smbh(params, deltah_data, steps=None, list_params=lisa_fisher.default_list_fisher_params_smbh, list_fixed_params=[], Lframe=False, fLow_from_22=True, **waveform_params):
    """Here we compute the Fisher matrix and CV-bias vector on the data grid.

    Use the frequency samples and TDI channels contained in `deltah_data`
    for both quantities.
    """

    waveform_params = copy.deepcopy(waveform_params)

    # Setting the number of recovery parameters
    Npar = len(list_params)

    LISAnoise = waveform_params.get('LISAnoise', pyLISAnoise.LISAnoiseSciRDv1)
    TDI = waveform_params.get('TDI', 'TDIAET')
    if not TDI == 'TDIAET':
        raise ValueError('Data is using TDI channels among A,E,T, must have TDI=TDIAET or TDI2AET')
    TDIrescaled = waveform_params.get('TDIrescaled', True)
    LISAconst = waveform_params.get('LISAconst', pyresponse.LISAconstProposal)

    # Converting the input parameters to the requested sky frame
    if not params.get('Lframe', False) and Lframe:
        params_base = lisatools.convert_SSBframe_to_Lframe(
                            params,
                            t0=waveform_params['t0'],
                            frozenLISA=waveform_params['frozenLISA'])
    elif params.get('Lframe', False) and not Lframe:
        params_base = lisatools.convert_Lframe_to_SSBframe(
                            params,
                            t0=waveform_params['t0'],
                            frozenLISA=waveform_params['frozenLISA'])
    else:
        params_base = params.copy()

    # Complete the mass and spin parameterizations
    params_base = pytools.complete_mass_params(params_base)
    params_base = pytools.complete_spin_params(params_base)
    m1 = params_base['m1']
    m2 = params_base['m2']
    M = params_base['M']
    Ms = M * pyconstants.MTSUN_SI

    # Identifying the TDI channels and translating their internal names
    tdi_channels = [k for k in deltah_data.keys() if not k == 'freq']
    tdi_translate = {'TDIA': 'chan1', 'TDIE': 'chan2', 'TDIT': 'chan3'}
    channels = [tdi_translate[k] for k in tdi_channels]
    deltah = dict([(tdi_translate[k], deltah_data[k]) for k in tdi_channels])

    # Generating the base recovery waveform
    wftdi_base = lisa.GenerateLISATDI_SMBH(params_base, **waveform_params)

    # Reading the modes and their frequency bounds from the base waveform
    modes = wftdi_base['modes']
    fLow_lm = dict([(lm, wftdi_base[lm]['freq'][0]) for lm in modes])
    fHigh_lm = dict([(lm, wftdi_base[lm]['freq'][-1]) for lm in modes])
    if fLow_from_22:
        fLow = fLow_lm[(2, 2)]
    else:
        fLow = np.min([fLow_lm[lm] for lm in modes])
    fHigh = np.max([fHigh_lm[lm] for lm in modes])

    # Using the injection grid for both the Fisher matrix and bias vector
    freqs = np.asarray(deltah_data['freq'])
    df = np.diff(freqs)

    # Fixing the internal grid for every finite-difference waveform
    if modes == [(2, 2)]:
        gridfreq = wftdi_base[(2, 2)]['freq'].copy()
    else:
        gridfreq = dict([(lm, wftdi_base[lm]['freq'].copy()) for lm in modes])
    waveform_params['gridfreq'] = gridfreq

    tdifreqseries_base = lisa.GenerateLISATDIFreqseries_SMBH(params_base, freqs, **waveform_params)

    # Selecting two independent mass parameters and two spin parameters
    list_mass_params = [p for p in list_params if p in pytools.list_mass_params]
    list_spin_params = [p for p in list_params if p in pytools.list_spin_params]
    list_fixed_mass_params = [p for p in list_fixed_params if p in pytools.list_mass_params]
    list_fixed_spin_params = [p for p in list_fixed_params if p in pytools.list_spin_params]
    list_pair_mass_params = list_mass_params + list_fixed_mass_params
    list_pair_spin_params = list_spin_params + list_fixed_spin_params
    if (not len(list_mass_params) + len(list_fixed_mass_params) == 2) or (not len(list_spin_params) + len(list_fixed_spin_params) == 2):
        raise ValueError('list_params + list_fixed_params must have 2 mass and spin params.')
    if list_pair_mass_params == ['q', 'eta'] or list_pair_mass_params == ['eta', 'q']:
        raise ValueError('Mass params pair (q,eta) is fully degenerate.')

    # Removing redundant mass and spin parameterizations
    params_base_pair_massspin = params_base.copy()
    for p in pytools.list_mass_params:
        params_base_pair_massspin.pop(p, None)
    for p in pytools.list_spin_params:
        params_base_pair_massspin.pop(p, None)
    for p in list_pair_mass_params + list_pair_spin_params:
        params_base_pair_massspin[p] = params_base[p]

    # Evaluating the LISA noise on the injection grid
    noise_evaluator = pyLISAnoise.initialize_noise(
        LISAnoise,
        TDI=TDI,
        TDIrescaled=TDIrescaled,
        LISAconst=LISAconst,
    )
    Sn1_vals, Sn2_vals, Sn3_vals = pyLISAnoise.evaluate_noise(
        noise_evaluator,
        freqs,
        TDI=TDI,
        TDIrescaled=TDIrescaled,
        LISAconst=LISAconst,
    )
    Snvals = {'chan1': Sn1_vals, 'chan2': Sn2_vals, 'chan3': Sn3_vals}

    # Setting the mass-dependent finite-difference steps
    steps_default = lisa_fisher.get_default_steps_smbh(M)
    if steps is None:
        steps = steps_default
    else:
        steps_tmp = copy.deepcopy(steps_default)
        steps_tmp.update(steps)
        steps = steps_tmp

    # Generating the finite-difference waveforms
    dh = {}
    for p in list_params:
        frac = lisa_fisher.dict_params_fractional[p]
        params_step = params_base_pair_massspin.copy()
        if frac:
            params_step[p] *= 1 + steps[p]
        else:
            params_step[p] += steps[p]
        params_step = pytools.complete_mass_params(params_step)
        params_step = pytools.complete_spin_params(params_step)

        tdifreqseries_step = lisa.GenerateLISATDIFreqseries_SMBH(params_step, freqs, **waveform_params)

        # Calculating the signal derivative for each TDI channel
        dh[p] = {}
        for chan in channels:
            dh[p][chan] = np.zeros_like(freqs, dtype=complex)
            for lm in modes:
                dh[p][chan] += (tdifreqseries_step[lm][chan] - tdifreqseries_base[lm][chan]) / steps[p]

    # Calculating beta_i on the injection grid
    beta = np.zeros(Npar, dtype=float)
    for i in range(Npar):
        pi = list_params[i]
        for chan in channels:
            integrand = np.real(dh[pi][chan] * np.conj(deltah[chan]) / Snvals[chan])
            beta[i] += 4 * np.sum(df * integrand[:-1])

    # Calculating Gamma_ij on the same grid and TDI channels
    fisher = np.zeros((Npar, Npar), dtype=float)
    for i in range(Npar):
        pi = list_params[i]
        for j in range(i, Npar):
            pj = list_params[j]
            val = 0.0
            for chan in channels:
                integrand = np.real(dh[pi][chan] * np.conj(dh[pj][chan]) / Snvals[chan])
                val += 4 * np.sum(df * integrand[:-1])
            fisher[i, j] = val
            fisher[j, i] = val

    # Restoring the physical scaling of parameters varied fractionally
    scales = np.ones(len(list_params))
    for i, p in enumerate(list_params):
        if lisa_fisher.dict_params_fractional[p]:
            scales[i] = 1.0 / params_base[p]
    scales_diag = np.diag(scales)

    beta = np.dot(scales_diag, beta)
    fisher = np.dot(scales_diag, np.dot(fisher, scales_diag))
    fisher = 0.5 * (fisher + fisher.T)

    try:
        cov = np.linalg.inv(fisher)
        covariance_method = 'inv'
    except np.linalg.LinAlgError:
        cov = np.linalg.pinv(fisher, rcond=1e-12)
        covariance_method = 'pinv'

    fishercov = {
        'fisher': fisher,
        'cov': cov,
        'list_params': list(list_params),
        'channels': list(tdi_channels),
        'n_freq': len(freqs),
        'freq_min': float(np.min(freqs)),
        'freq_max': float(np.max(freqs)),
        'freq_grid_source': 'deltah_data["freq"]',
        'covariance_method': covariance_method,
    }

    return fishercov, beta


In [7]:
def format_mass_title(mass_value, use_latex=True):
    """Formatting a numerical total mass for figure titles."""
    mass_value = float(mass_value)
    exponent = int(np.floor(np.log10(mass_value)))
    mantissa = mass_value / (10.0 ** exponent)

    if np.isclose(mantissa, 1.0):
        if use_latex:
            return rf"$M=10^{{{exponent}}}\,M_{{\odot}}$"
        return f"1e{exponent} Msun"

    if use_latex:
        return rf"$M={mantissa:g}\times10^{{{exponent}}}\,M_{{\odot}}$"
    return f"{mantissa:g}e{exponent} Msun"


## 2. Three-mass FTI recovery configuration

Matching the injection notebook’s tag, mass labels, seven deviation IDs, waveform modes, and output structure before starting the recovery calculation.


In [8]:
# Matching the tag used by the three-mass injection notebook
deviation_tag = "domega_dtau_scan_M1e5_M1e6_M1e7_extra"

Syst_dir = os.path.expanduser(
    "/pbs/home/a/amahdi/pySEOBNR_deviation/Injection set"
)
Input_scan_dir = os.path.join(Syst_dir, "pSEOB_" + deviation_tag)
Run_dir = os.path.join(
    Syst_dir,
    "SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_"
    + deviation_tag,
)

# Setting the same mass array and labels used during injection generation
MASS_VALUES = np.array([1e5, 1e6, 1e7], dtype=float)
MASS_LABELS = ["M1e5", "M1e6", "M1e7"]
MASS_VALUE_BY_LABEL = dict(zip(MASS_LABELS, MASS_VALUES))
MASS_TITLE_BY_LABEL = {
    label: format_mass_title(value)
    for value, label in zip(MASS_VALUES, MASS_LABELS)
}

inj_ids = list(range(7))
expected_deviation_values = np.array(
    [0.1, 0.01, 0.001, 0.0001, 0.0, 0.05, 0.02],
    dtype=float,
)

wf_rec = "SEOB"
if wf_rec == "SEOB":
    wf_name = "SEOBNRv5HMROM"
elif wf_rec == "XHM":
    wf_name = "PhenomXHM"
else:
    raise ValueError(f"Unknown wf_rec = {wf_rec}")

output_tag = "rom_pseob_ratio_phase_" + deviation_tag
RECOVERY_ROOT = os.path.join(Run_dir, "run_data", "bias_syst_recovery")
os.makedirs(RECOVERY_ROOT, exist_ok=True)

# Locate and validate one parameter file for each mass
PARAMS_FILES = {}
for mass_value, mass_label in zip(MASS_VALUES, MASS_LABELS):
    params_filename = (
        "params_injection_set_" + deviation_tag + "_" + mass_label + ".h5"
    )
    candidates = [
        os.path.join(Run_dir, params_filename),
        os.path.join(Input_scan_dir, params_filename),
    ]

    params_file = next(
        (candidate for candidate in candidates if os.path.exists(candidate)),
        None,
    )
    if params_file is None:
        raise FileNotFoundError(
            f"No parameter file was found for {mass_label}.\n"
            + "\n".join(candidates)
            + "\nRun the three-mass injection notebook first."
        )

    with h5py.File(params_file, "r") as hf:
        if "M" not in hf or len(hf["M"]) < len(inj_ids):
            raise ValueError(
                f"The parameter file for {mass_label} does not contain "
                f"the required {len(inj_ids)} rows: {params_file}"
            )

        stored_masses = np.asarray(hf["M"][:len(inj_ids)], dtype=float)
        if not np.allclose(stored_masses, mass_value):
            raise ValueError(
                f"Mass mismatch in {params_file}: expected {mass_value}, "
                f"found {stored_masses}."
            )

        if "d_omega" in hf:
            stored_domega = np.asarray(
                hf["d_omega"][:len(inj_ids)],
                dtype=float,
            )
            if not np.allclose(stored_domega, expected_deviation_values):
                raise ValueError(
                    f"d_omega values do not match the seven-point scan in "
                    f"{params_file}."
                )

        if "d_tau" in hf:
            stored_dtau = np.asarray(
                hf["d_tau"][:len(inj_ids)],
                dtype=float,
            )
            if not np.allclose(stored_dtau, expected_deviation_values):
                raise ValueError(
                    f"d_tau values do not match the seven-point scan in "
                    f"{params_file}."
                )

    PARAMS_FILES[mass_label] = params_file
    os.makedirs(os.path.join(RECOVERY_ROOT, mass_label), exist_ok=True)

generation_manifest = os.path.join(
    Run_dir,
    f"generation_manifest_{deviation_tag}.json",
)

print("Run_dir:", Run_dir)
print("Recovery root:", RECOVERY_ROOT)
print("Mass folders:", MASS_LABELS)
print("Injection IDs:", inj_ids)
print("Parameter files:")
for mass_label, params_file in PARAMS_FILES.items():
    print(f"  {mass_label}: {params_file}")
print("Generation manifest exists?", os.path.exists(generation_manifest))


Run_dir: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra
Recovery root: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/run_data/bias_syst_recovery
Mass folders: ['M1e5', 'M1e6', 'M1e7']
Injection IDs: [0, 1, 2, 3, 4, 5, 6]
Parameter files:
  M1e5: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/params_injection_set_domega_dtau_scan_M1e5_M1e6_M1e7_extra_M1e5.h5
  M1e6: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/params_injection_set_domega_dtau_scan_M1e5_M1e6_M1e7_extra_M1e6.h5
  M1e7: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_

In [9]:
# Setting the recovery waveform and LISA response
waveform_params_base_recovery = {
    "minf": 1e-5,
    "maxf": 0.5,
    "t0": 0.0,
    "timetomerger_max": 1.0,
    "fend": None,
    "tmin": None,
    "tmax": None,
    "phiref": 0.0,
    "fref_for_phiref": 0.0,
    "tref": 0.0,
    "fref_for_tref": 0.0,
    "force_phiref_fref": True,
    "toffset": 0.0,
    "modes": [(2, 2), (2, 1), (3, 3), (4, 4), (4, 3), (5, 5)],
    "TDI": "TDIAET",
    "acc": 0.0001,
    "order_fresnel_stencil": 0,
    "approximant": "SEOBNRv5HMROM",
    "LISAconst": "Proposal",
    "responseapprox": "full",
    "frozenLISA": False,
    "TDIrescaled": True,
    "LISAnoise": {
        "InstrumentalNoise": "SciRDv1",
        "WDbackground": True,
        "WDduration": 4.0,
        "lowf_add_pm_noise_f0": 0.0,
        "lowf_add_pm_noise_alpha": 2.0,
    },
}

BASE_GR_PARAM = [
    "Mchirp", "q", "chiPN", "chim", "Deltat",
    "dist", "inc", "phi", "lambda", "beta", "psi",
]
alignment_param = ["Deltat", "phi", "psi"]

# Activating one FTI parameter at a time during the Fisher calculation to avoid degeneracy between them
FTI_PARAMS_TO_SCAN = [
    "fti_dchiMinus2v",
    "fti_dchi0v",
    "fti_dchi1v",
    "fti_dchi2v",
    "fti_dchi3v",
    "fti_dchi4v",
    "fti_dchi5lv",
    "fti_dchi6v",
    "fti_dchi6lv",
    "fti_dchi7v",
    "fti_dkappa_S",
    "fti_dxi0v",
]

# Keep every FTI recovery value at zero for the GR-template bias test
FTI_PARAM_VALUES = {param: 0.0 for param in FTI_PARAMS_TO_SCAN}

Ntries = 10
Deltat_max = 200.0

print("Recovery approximant:", waveform_params_base_recovery["approximant"])
print("Base GR parameters:", BASE_GR_PARAM)
print("FTI parameters scanned one at a time:", FTI_PARAMS_TO_SCAN)


Recovery approximant: SEOBNRv5HMROM
Base GR parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi']
FTI parameters scanned one at a time: ['fti_dchiMinus2v', 'fti_dchi0v', 'fti_dchi1v', 'fti_dchi2v', 'fti_dchi3v', 'fti_dchi4v', 'fti_dchi5lv', 'fti_dchi6v', 'fti_dchi6lv', 'fti_dchi7v', 'fti_dkappa_S', 'fti_dxi0v']


In [10]:
def get_mass_run_data_dir(mass_label):
    """Returning the recovery output directory for one mass."""
    return os.path.join(RECOVERY_ROOT, mass_label)


def get_injection_data_file(mass_label, inj_id):
    """Returning the summed multiplier-injection file for one mass and ID."""
    return os.path.join(
        Run_dir,
        mass_label,
        "data_" + output_tag + "_" + mass_label + "_id_" + str(inj_id) + ".h5"
    )


def get_injection_datalm_file(mass_label, inj_id):
    """Returning the mode-resolved injection file for one mass and ID."""
    return os.path.join(
        Run_dir,
        mass_label,
        "datalm_" + output_tag + "_" + mass_label + "_id_" + str(inj_id) + ".h5"
    )


def get_multiplier_file(mass_label, inj_id):
    """Returning the pSEOB multiplier file for one mass and ID."""
    return os.path.join(
        Run_dir,
        mass_label,
        "multiplier_pseob_ratio_phase_" + deviation_tag + "_" + mass_label + "_id_" + str(inj_id) + ".h5"
    )


def load_ratio_wrapper_injection(mass_label, inj_id, params_file=None):
    """Loading one mass-specific multiplier injection and its parameters."""
    if params_file is None:
        params_file = PARAMS_FILES[mass_label]

    data_file = get_injection_data_file(mass_label, inj_id)
    datalm_file = get_injection_datalm_file(mass_label, inj_id)
    multiplier_file = get_multiplier_file(mass_label, inj_id)

    if not os.path.exists(params_file):
        raise FileNotFoundError(f"Missing params_file: {params_file}")
    if not os.path.exists(data_file):
        raise FileNotFoundError(f"Missing ratio-wrapper data_file for inj_id={inj_id}: {data_file}")

    print("Loading ratio-wrapper injection data from:")
    print(data_file)
    print("Mode-by-mode datalm file exists?", os.path.exists(datalm_file), datalm_file)
    print("Multiplier file exists?", os.path.exists(multiplier_file), multiplier_file)

    with h5py.File(params_file, 'r') as hf:
        params = {
            'M': float(hf['M'][inj_id]),
            'q': float(hf['q'][inj_id]),
            'chi1': float(hf['chi1'][inj_id]),
            'chi2': float(hf['chi2'][inj_id]),
            'dist': float(hf['dist'][inj_id]),
            'inc': float(hf['inc'][inj_id]),
            'phi': float(hf['phi'][inj_id]),
            'lambda': float(hf['lambda'][inj_id]),
            'beta': float(hf['beta'][inj_id]),
            'psi': float(hf['psi'][inj_id]),
            'Deltat': float(hf['Deltat'][inj_id]),
            'Lframe': bool(hf['Lframe'][:][0]),
        }

        # Keeping the injected deviations as labels for plots and checks
        if 'd_omega' in hf:
            params['d_omega'] = float(hf['d_omega'][inj_id])
        if 'd_tau' in hf:
            params['d_tau'] = float(hf['d_tau'][inj_id])

    print("Loaded parameter labels:")
    print("  d_omega =", params.get('d_omega', None))
    print("  d_tau   =", params.get('d_tau', None))

    tdidata = inference.load_data_TDIAET(data_file)
    return params, tdidata


def make_fti_recovery_inputs(params0, fti_param, fti_value=0.0):
    params = params0.copy()

    # Removing deviation labels before generating the ROM recovery template
    params.pop('d_omega', None)
    params.pop('d_tau', None)

    # Removing every FTI key because lisabeta checks key presence
    for p in FTI_PARAMS_TO_SCAN:
        params.pop(p, None)

    # Adding only the active FTI parameter for the current case
    params[fti_param] = fti_value

    params_complete_mass = pytools.complete_mass_params(params)

    waveform_params_local = copy.deepcopy(waveform_params_base_recovery)
    waveform_params_local.pop("tgr_params", None)

    waveform_params_local["tgr_params"] = {
        "FTI": {
            "fti_params": [fti_param],
            "Mchirp_inj": params_complete_mass["Mchirp"],
            "dxi0v_inj": 0.0
        }
    }

    list_params = BASE_GR_PARAM + [fti_param]

    return params, waveform_params_local, list_params


def build_recovery_tdi_on_injection_grid(injection_align, tdidata, waveform_params):
    """Generating the ROM recovery template on the injection frequency grid."""
    freq_grid = tdidata['freq']
    tdifreqseries = lisa.GenerateLISATDIFreqseries_SMBH(injection_align, freq_grid, **waveform_params)

    tdi_align = {
        'freq': freq_grid.copy(),
        'TDIA': np.zeros(len(freq_grid), dtype=complex),
        'TDIE': np.zeros(len(freq_grid), dtype=complex),
        'TDIT': np.zeros(len(freq_grid), dtype=complex),
    }

    for lm in tdifreqseries['modes']:
        tdi_align['TDIA'] += tdifreqseries[lm]['chan1']
        tdi_align['TDIE'] += tdifreqseries[lm]['chan2']
        tdi_align['TDIT'] += tdifreqseries[lm]['chan3']

    return tdi_align


def process_nelder_mead_result(best_x, Deltat_max=200.0):
    # Convert the scaled optimizer variables back to physical parameters
    deltat_true = best_x[0] * Deltat_max
    phi_true = (best_x[1] + np.pi) % (2 * np.pi) - np.pi
    psi_true = best_x[2] % np.pi
    return {
        'Deltat': deltat_true,
        'phi': phi_true,
        'psi': psi_true,
    }


def run_nelder_mead_alignment(params, tdidata, waveform_params, Ntries=50, Deltat_max=200.0, seed=None):
    # Run the direct Nelder-Mead likelihood maximization
    if seed is not None:
        rng = np.random.default_rng(seed)
        Deltat_tries = rng.uniform(-Deltat_max, Deltat_max, size=Ntries)
        phi_tries = rng.uniform(-np.pi, np.pi, size=Ntries)
        psi_tries = rng.uniform(0.0, np.pi, size=Ntries)
    else:
        Deltat_tries = np.random.uniform(-Deltat_max, Deltat_max, size=Ntries)
        phi_tries = np.random.uniform(-np.pi, np.pi, size=Ntries)
        psi_tries = np.random.uniform(0.0, np.pi, size=Ntries)

    likelihood = lisa.LikelihoodLISASMBH_Data(params, data=tdidata, **waveform_params)

    def f_opt(x):
        params_NM = params.copy()
        params_NM['Deltat'] = Deltat_max * x[0]
        params_NM['phi'] = x[1]
        params_NM['psi'] = x[2]

        if np.abs(x[0]) > 1.0:
            return np.inf
        return -likelihood.lnL(params_NM, ngrid=128, **waveform_params)

    scale = 0.5
    res_neldermead = {}

    for j in tqdm(range(Ntries), desc='Nelder-Mead starts', leave=False):
        x0 = np.array([Deltat_tries[j] / Deltat_max, phi_tries[j], psi_tries[j]])
        initial_simplex = np.array([
            x0,
            x0 + scale * np.array([1, 0, 0]),
            x0 + scale * np.array([0, 1, 0]),
            x0 + scale * np.array([0, 0, 1]),
        ])
        res_neldermead[j] = scipy.optimize.minimize(
            f_opt,
            x0,
            method='Nelder-Mead',
            options={'initial_simplex': initial_simplex, 'return_all': True},
        )

    best_j = min(res_neldermead, key=lambda k: res_neldermead[k].fun)
    best_result = res_neldermead[best_j]
    ML_values = process_nelder_mead_result(np.array(best_result.x), Deltat_max=Deltat_max)

    return best_j, best_result, ML_values, res_neldermead


def compute_match_mismatch(tdidata, tdi_align, waveform_params):
    # Calculating the TDI overlap, match, and mismatch
    freq_grid = tdidata['freq']
    lisa_psd = pyLISAnoise.evaluate_AET_psd(freq_grid, TDIT=True, **waveform_params)

    overlap_12 = overlap_AET_data(tdidata, tdi_align, lisa_psd)
    overlap_11 = overlap_AET_data(tdidata, tdidata, lisa_psd)
    overlap_22 = overlap_AET_data(tdi_align, tdi_align, lisa_psd)

    norm_factor = np.sqrt(overlap_11) * np.sqrt(overlap_22)
    match = np.abs(overlap_12) / norm_factor
    mismatch = 1.0 - match

    return match, mismatch, lisa_psd


def save_recovery_arrays(case_dir, mass_label, inj_id, fti_param, tdidata, tdi_align, residual_align):
    # Saving the recovery template and residual arrays for later inspection
    os.makedirs(case_dir, exist_ok=True)
    out_file = os.path.join(case_dir, f'recovery_arrays_{fti_param}_{mass_label}_id_{inj_id}.h5')
    with h5py.File(out_file, 'w') as hf:
        hf.attrs['fti_param'] = fti_param
        hf.attrs['inj_id'] = inj_id
        hf.attrs['mass_label'] = mass_label
        hf.attrs['M'] = float(MASS_VALUE_BY_LABEL[mass_label])
        hf.create_dataset('freq', data=tdidata['freq'])
        for chan in ['TDIA', 'TDIE']:
            hf.create_dataset('injection_' + chan, data=tdidata[chan])
            hf.create_dataset('template_' + chan, data=tdi_align[chan])
            hf.create_dataset('residual_' + chan, data=residual_align[chan])
        if 'TDIT' in tdi_align:
            hf.create_dataset('template_TDIT', data=tdi_align['TDIT'])
    return out_file


In [11]:
# Check every mass-specific injection path before starting the expensive scan
print("Run_dir used by this notebook:", Run_dir)
print("Run_dir exists?", os.path.exists(Run_dir))

missing_injection_files = []
for mass_label in MASS_LABELS:
    print("\n" + "=" * 80)
    print(f"Checking {mass_label}")
    print("Parameter file:", PARAMS_FILES[mass_label])

    for inj_id in inj_ids:
        data_file = get_injection_data_file(mass_label, inj_id)
        datalm_file = get_injection_datalm_file(mass_label, inj_id)
        multiplier_file = get_multiplier_file(mass_label, inj_id)

        print(f"inj_id={inj_id}:")
        print("  data exists?      ", os.path.exists(data_file), data_file)
        print("  datalm exists?    ", os.path.exists(datalm_file), datalm_file)
        print("  multiplier exists?", os.path.exists(multiplier_file), multiplier_file)

        for file_path in (data_file, datalm_file, multiplier_file):
            if not os.path.exists(file_path):
                missing_injection_files.append(file_path)

if missing_injection_files:
    preview = "\n".join(missing_injection_files[:10])
    raise FileNotFoundError(
        f"Missing {len(missing_injection_files)} injection files.\n"
        f"First missing paths:\n{preview}\n"
        "Run the complete three-mass injection notebook first."
    )


Run_dir used by this notebook: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra
Run_dir exists? True

Checking M1e5
Parameter file: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/params_injection_set_domega_dtau_scan_M1e5_M1e6_M1e7_extra_M1e5.h5
inj_id=0:
  data exists?       True /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/M1e5/data_rom_pseob_ratio_phase_domega_dtau_scan_M1e5_M1e6_M1e7_extra_M1e5_id_0.h5
  datalm exists?     True /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/M1e5/datalm_rom_pseob_ratio_phase_domega_dtau_scan_M1e5_M1e6_M1e7_extra_M1e5_id_0.h5
  multiplier exists? Tru

In [ ]:
# Running one alignment per mass and injection ID, followed by the FTI scan
all_results = {mass_label: {} for mass_label in MASS_LABELS}
alignment_results = {mass_label: {} for mass_label in MASS_LABELS}

# Using reproducible but mass-dependent alignment starts
seed_alignment = 1234
reference_fti_param = FTI_PARAMS_TO_SCAN[0]

for mass_index, (mass_value, mass_label) in enumerate(
    zip(MASS_VALUES, MASS_LABELS)
):
    mass_run_data_dir = get_mass_run_data_dir(mass_label)
    os.makedirs(mass_run_data_dir, exist_ok=True)

    print("\n" + "#" * 90)
    print(f"STARTING MASS: {mass_label} ({mass_value:.6e} Msun)")
    print("#" * 90)

    for inj_id in inj_ids:
        all_results[mass_label][inj_id] = {}

        print("\n" + "#" * 90)
        print(f"STARTING {mass_label}, inj_id={inj_id}")
        print("#" * 90)

        # Loading the current injection only once
        params0, tdidata = load_ratio_wrapper_injection(
            mass_label=mass_label,
            inj_id=inj_id,
        )

        if not np.isclose(params0["M"], mass_value):
            raise ValueError(
                f"Loaded mass mismatch for {mass_label}, inj_id={inj_id}: "
                f"expected {mass_value}, found {params0['M']}"
            )

        # Building one GR recovery setup for the alignment
        params_ref, waveform_params_ref, list_params_ref = (
            make_fti_recovery_inputs(
                params0,
                reference_fti_param,
                fti_value=0.0,
            )
        )

        print("\n" + "=" * 80)
        print(f"Running Nelder-Mead once for {mass_label}, inj_id={inj_id}")
        print(f"Reference FTI parameter: {reference_fti_param}")

        seed_local = None
        if seed_alignment is not None:
            seed_local = (
                seed_alignment
                + 100_000 * mass_index
                + 1_000 * int(inj_id)
            )

        best_j, best_result, ML_values, res_neldermead = (
            run_nelder_mead_alignment(
                params_ref,
                tdidata,
                waveform_params_ref,
                Ntries=Ntries,
                Deltat_max=Deltat_max,
                seed=seed_local,
            )
        )

        print(f"Best trial: {best_j}")
        print(f"Minimum value (-lnL): {best_result.fun}")
        print("Best alignment values:", ML_values)

        # Building the aligned GR waveform and residual once
        injection_align_ref = params_ref.copy()
        for param in alignment_param:
            injection_align_ref[param] = ML_values[param]

        tdi_align = build_recovery_tdi_on_injection_grid(
            injection_align_ref,
            tdidata,
            waveform_params_ref,
        )

        residual_align = {"freq": tdidata["freq"]}
        for channel in ("TDIA", "TDIE"):
            residual_align[channel] = tdi_align[channel] - tdidata[channel]

        match, mismatch, lisa_psd = compute_match_mismatch(
            tdidata,
            tdi_align,
            waveform_params_ref,
        )

        print(f"Match: {match:.10f}")
        print(f"Mismatch: {mismatch:.10e}")

        alignment_results[mass_label][inj_id] = {
            "mass_label": mass_label,
            "M": float(mass_value),
            "mass_name": MASS_TITLE_BY_LABEL[mass_label],
            "inj_id": inj_id,
            "d_omega": params0.get("d_omega"),
            "d_tau": params0.get("d_tau"),
            "params0": params0,
            "tdidata": tdidata,
            "reference_fti_param": reference_fti_param,
            "reference_params": params_ref,
            "reference_waveform_params": waveform_params_ref,
            "best_j": best_j,
            "best_result": best_result,
            "ML_values": ML_values,
            "injection_align_ref": injection_align_ref,
            "tdi_align": tdi_align,
            "residual_align": residual_align,
            "match": match,
            "mismatch": mismatch,
            "lisa_psd": lisa_psd,
            "res_neldermead": res_neldermead,
        }

        # Reusing the aligned residual while changing only the active FTI parameter
        for fti_param in FTI_PARAMS_TO_SCAN:
            fti_value = FTI_PARAM_VALUES.get(fti_param, 0.0)

            params, waveform_params, list_params = make_fti_recovery_inputs(
                params0,
                fti_param,
                fti_value=fti_value,
            )

            print("\n" + "=" * 80)
            print(f"Mass: {mass_label}, inj_id={inj_id}")
            print(f"FTI CV-bias case: {fti_param}={fti_value}")
            print("Using the already aligned residual.")

            injection_align = params.copy()
            for param in alignment_param:
                injection_align[param] = ML_values[param]

            injection_align_complete = pytools.complete_mass_params(
                injection_align.copy()
            )

            fishercov, beta_vec = fisher_beta_data_grid_smbh(
                injection_align_complete,
                residual_align,
                list_params=list_params,
                Lframe=True,
                **waveform_params,
            )

            deltatheta = np.dot(fishercov["cov"], -beta_vec)
            scales = {
                param: np.sqrt(fishercov["cov"][index, index])
                for index, param in enumerate(list_params)
            }
            cv_bias_sigma = {
                param: deltatheta[index] / scales[param]
                for index, param in enumerate(list_params)
            }

            case_dir = os.path.join(
                mass_run_data_dir,
                f"id_{inj_id}",
                fti_param,
            )
            arrays_file = save_recovery_arrays(
                case_dir,
                mass_label,
                inj_id,
                fti_param,
                tdidata,
                tdi_align,
                residual_align,
            )

            i_fti = list_params.index(fti_param)
            print("Recovered CV bias in FTI:", deltatheta[i_fti])
            print("Recovered in sigma units:", cv_bias_sigma[fti_param])
            print("Saved arrays:", arrays_file)

            all_results[mass_label][inj_id][fti_param] = {
                "mass_label": mass_label,
                "M": float(mass_value),
                "mass_name": MASS_TITLE_BY_LABEL[mass_label],
                "inj_id": inj_id,
                "d_omega": params0.get("d_omega"),
                "d_tau": params0.get("d_tau"),
                "fti_param": fti_param,
                "fti_value": fti_value,
                "list_params": list_params,
                "params": params,
                "tdidata": tdidata,
                "waveform_params": waveform_params,
                "best_j": best_j,
                "best_result": best_result,
                "ML_values": ML_values,
                "injection_align": injection_align,
                "res_neldermead": res_neldermead,
                "tdi_align": tdi_align,
                "residual_align": residual_align,
                "match": match,
                "mismatch": mismatch,
                "lisa_psd": lisa_psd,
                "beta": beta_vec,
                "fishercov": fishercov,
                "deltatheta": deltatheta,
                "scales": scales,
                "cv_bias_sigma": cv_bias_sigma,
                "arrays_file": arrays_file,
            }

print("Finished masses:", list(all_results.keys()))
for mass_label, results_by_inj in all_results.items():
    print(f"{mass_label}: finished injection IDs:", list(results_by_inj.keys()))



##########################################################################################
STARTING MASS: M1e5 (1.000000e+05 Msun)
##########################################################################################

##########################################################################################
STARTING M1e5, inj_id=0
##########################################################################################
Loading ratio-wrapper injection data from:
/pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/M1e5/data_rom_pseob_ratio_phase_domega_dtau_scan_M1e5_M1e6_M1e7_extra_M1e5_id_0.h5
Mode-by-mode datalm file exists? True /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/M1e5/datalm_rom_pseob_ratio_phase_domega_dtau_scan_M1e5_M1e6_M1e7_extra_M1e5_id_0.h5
Multiplier file exists?

Best trial: 5
Minimum value (-lnL): 230.98193890209706
Best alignment values: {'Deltat': np.float64(-0.3438284605855625), 'phi': np.float64(-0.36589225075423215), 'psi': np.float64(2.785939845414039)}
Match: 0.9970484787
Mismatch: 2.9515212526e-03

Mass: M1e5, inj_id=0
FTI CV-bias case: fti_dchiMinus2v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: 6.014675484307365e-07
Recovered in sigma units: 2.076928095388659
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/run_data/bias_syst_recovery/M1e5/id_0/fti_dchiMinus2v/recovery_arrays_fti_dchiMinus2v_M1e5_id_0.h5

Mass: M1e5, inj_id=0
FTI CV-bias case: fti_dchi0v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: 0.00028699918130925073
Recovered in sigma units: 2.2364269748734626
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_co

Best trial: 5
Minimum value (-lnL): 451.3231374356877
Best alignment values: {'Deltat': np.float64(-0.07325735702976512), 'phi': np.float64(-0.4598759373538064), 'psi': np.float64(2.8180665247035375)}
Match: 0.9949936399
Mismatch: 5.0063601081e-03

Mass: M1e5, inj_id=1
FTI CV-bias case: fti_dchiMinus2v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: 2.057248318327897e-06
Recovered in sigma units: 7.037679928907872
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/run_data/bias_syst_recovery/M1e5/id_1/fti_dchiMinus2v/recovery_arrays_fti_dchiMinus2v_M1e5_id_1.h5

Mass: M1e5, inj_id=1
FTI CV-bias case: fti_dchi0v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: 0.0006227397758976341
Recovered in sigma units: 4.864842160417892
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_cond

Best trial: 6
Minimum value (-lnL): 0.6193034430846884
Best alignment values: {'Deltat': np.float64(-0.002095417389282243), 'phi': np.float64(0.7059694907584202), 'psi': np.float64(1.1976847349901443)}
Match: 0.9992169771
Mismatch: 7.8302287767e-04

Mass: M1e5, inj_id=2
FTI CV-bias case: fti_dchiMinus2v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: -8.077424621781865e-08
Recovered in sigma units: -0.28012472601078253
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/run_data/bias_syst_recovery/M1e5/id_2/fti_dchiMinus2v/recovery_arrays_fti_dchiMinus2v_M1e5_id_2.h5

Mass: M1e5, inj_id=2
FTI CV-bias case: fti_dchi0v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: -3.454395760374505e-05
Recovered in sigma units: -0.26852292297614405
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_d

Best trial: 5
Minimum value (-lnL): 0.006389158707125284
Best alignment values: {'Deltat': np.float64(-0.000646723710458756), 'phi': np.float64(0.7005890062625331), 'psi': np.float64(1.199734488775239)}
Match: 0.9992244599
Mismatch: 7.7554006817e-04

Mass: M1e5, inj_id=3
FTI CV-bias case: fti_dchiMinus2v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: -8.280522451155297e-09
Recovered in sigma units: -0.028703488891041323
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/run_data/bias_syst_recovery/M1e5/id_3/fti_dchiMinus2v/recovery_arrays_fti_dchiMinus2v_M1e5_id_3.h5

Mass: M1e5, inj_id=3
FTI CV-bias case: fti_dchi0v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: -3.474193621302152e-06
Recovered in sigma units: -0.027013352353265054
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_wit

...

Best trial: 2
Minimum value (-lnL): 3993.1091373949357
Best alignment values: {'Deltat': np.float64(-3.708421327511962), 'phi': np.float64(0.7105092359341545), 'psi': np.float64(1.212197299912754)}
Match: 0.9997199101
Mismatch: 2.8008989943e-04

Mass: M1e7, inj_id=1
FTI CV-bias case: fti_dchiMinus2v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: -2.4409270937623658e-05
Recovered in sigma units: -2.35495101527821
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/run_data/bias_syst_recovery/M1e7/id_1/fti_dchiMinus2v/recovery_arrays_fti_dchiMinus2v_M1e7_id_1.h5

Mass: M1e7, inj_id=1
FTI CV-bias case: fti_dchi0v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: 0.0006185800459345137
Recovered in sigma units: 1.680572138721279
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condi

Best trial: 2
Minimum value (-lnL): 38.81220058635881
Best alignment values: {'Deltat': np.float64(-0.3462805439578855), 'phi': np.float64(0.7010137159201615), 'psi': np.float64(1.201197566561051)}
Match: 0.9999965094
Mismatch: 3.4905893277e-06

Mass: M1e7, inj_id=2
FTI CV-bias case: fti_dchiMinus2v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: -2.7488171814318515e-06
Recovered in sigma units: -0.26512338780097094
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/run_data/bias_syst_recovery/M1e7/id_2/fti_dchiMinus2v/recovery_arrays_fti_dchiMinus2v_M1e7_id_2.h5

Mass: M1e7, inj_id=2
FTI CV-bias case: fti_dchi0v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: 4.6932103156664164e-05
Recovered in sigma units: 0.12746958878445563
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom

Best trial: 6
Minimum value (-lnL): 0.3871865952918466
Best alignment values: {'Deltat': np.float64(-0.00935121893776424), 'phi': np.float64(0.7000988838545426), 'psi': np.float64(1.2001217729006006)}
Match: 0.9999992097
Mismatch: 7.9028185651e-07

Mass: M1e7, inj_id=3
FTI CV-bias case: fti_dchiMinus2v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: -2.778845717086332e-07
Recovered in sigma units: -0.02680121443557357
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/run_data/bias_syst_recovery/M1e7/id_3/fti_dchiMinus2v/recovery_arrays_fti_dchiMinus2v_M1e7_id_3.h5

Mass: M1e7, inj_id=3
FTI CV-bias case: fti_dchi0v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: 4.542936325489593e-06
Recovered in sigma units: 0.012338469792380558
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_den

Best trial: 5
Minimum value (-lnL): 1.9464609021185417e-05
Best alignment values: {'Deltat': np.float64(-9.224790322883245e-05), 'phi': np.float64(0.6999992270027264), 'psi': np.float64(1.199998621699104)}
Match: 0.9999992369
Mismatch: 7.6313826713e-07

Mass: M1e7, inj_id=4
FTI CV-bias case: fti_dchiMinus2v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: -4.633978653368327e-14
Recovered in sigma units: -4.4693325210501835e-09
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/run_data/bias_syst_recovery/M1e7/id_4/fti_dchiMinus2v/recovery_arrays_fti_dchiMinus2v_M1e7_id_4.h5

Mass: M1e7, inj_id=4
FTI CV-bias case: fti_dchi0v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: -4.380508159847326e-12
Recovered in sigma units: -1.18972829243065e-08
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phas

Best trial: 3
Minimum value (-lnL): 111103.56705890826
Best alignment values: {'Deltat': np.float64(-20.787858497033582), 'phi': np.float64(0.7608586503517518), 'psi': np.float64(1.2651586577725462)}
Match: 0.9924141866
Mismatch: 7.5858134053e-03

Mass: M1e7, inj_id=5
FTI CV-bias case: fti_dchiMinus2v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: -3.423659476477908e-05
Recovered in sigma units: -3.307624427980412
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e5_M1e6_M1e7_extra/run_data/bias_syst_recovery/M1e7/id_5/fti_dchiMinus2v/recovery_arrays_fti_dchiMinus2v_M1e7_id_5.h5

Mass: M1e7, inj_id=5
FTI CV-bias case: fti_dchi0v=0.0
Using the already aligned residual.
Recovered CV bias in FTI: 0.006733904965472909
Recovered in sigma units: 18.32071108096891
Saved arrays: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_cond

In [ ]:
# Collect the alignment and mismatch summary
summary_rows = []

for mass_label, results_by_inj in all_results.items():
    for inj_id, results_by_fti in results_by_inj.items():
        for fti_param, result in results_by_fti.items():
            summary_rows.append({
                "mass_label": mass_label,
                "M": result["M"],
                "inj_id": inj_id,
                "d_omega": result.get("d_omega"),
                "d_tau": result.get("d_tau"),
                "fti_param": fti_param,
                "fti_value": result["fti_value"],
                "match": result["match"],
                "mismatch": result["mismatch"],
                "best_trial": result["best_j"],
                "minus_lnL_min": result["best_result"].fun,
                "ML_Deltat": result["ML_values"]["Deltat"],
                "ML_phi": result["ML_values"]["phi"],
                "ML_psi": result["ML_values"]["psi"],
                "arrays_file": result["arrays_file"],
            })

summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(
    RECOVERY_ROOT,
    "summary_alignment_mismatch_all_masses_all_fti_cases.csv",
)
summary_df.to_csv(summary_csv, index=False)
print("Saved:", summary_csv)

for mass_label in MASS_LABELS:
    mass_summary = summary_df[summary_df["mass_label"] == mass_label]
    mass_csv = os.path.join(
        get_mass_run_data_dir(mass_label),
        f"summary_alignment_mismatch_{mass_label}.csv",
    )
    mass_summary.to_csv(mass_csv, index=False)
    print("Saved:", mass_csv)

summary_df


### CV-bias tables

Collecting the displacement, statistical uncertainty, and normalized bias for every recovery parameter, mass, injection ID, and active FTI case.


In [ ]:
bias_rows = []

for mass_label, results_by_inj in all_results.items():
    for inj_id, results_by_fti in results_by_inj.items():
        for fti_param, result in results_by_fti.items():
            list_params = result["list_params"]

            for index, param in enumerate(list_params):
                bias_rows.append({
                    "mass_label": mass_label,
                    "M": result["M"],
                    "inj_id": inj_id,
                    "d_omega": result.get("d_omega"),
                    "d_tau": result.get("d_tau"),
                    "fti_case": fti_param,
                    "fti_value": result["fti_value"],
                    "parameter": param,
                    "Delta_theta_over_sigma": result["cv_bias_sigma"][param],
                    "Delta_theta": result["deltatheta"][index],
                    "sigma": result["scales"][param],
                    "is_active_fti_parameter": param == fti_param,
                })

bias_df = pd.DataFrame(bias_rows)
bias_csv = os.path.join(
    RECOVERY_ROOT,
    "cv_bias_sigma_all_masses_all_fti_cases.csv",
)
bias_df.to_csv(bias_csv, index=False)
print("Saved:", bias_csv)

for mass_label in MASS_LABELS:
    mass_bias = bias_df[bias_df["mass_label"] == mass_label]
    mass_csv = os.path.join(
        get_mass_run_data_dir(mass_label),
        f"cv_bias_sigma_{mass_label}.csv",
    )
    mass_bias.to_csv(mass_csv, index=False)
    print("Saved:", mass_csv)

bias_df


In [ ]:
# Plot every recovery parameter for each mass, injection, and FTI case
mpl.rcParams["font.family"] = "serif"
plt.close("all")

for mass_label, results_by_inj in all_results.items():
    mass_name = MASS_TITLE_BY_LABEL[mass_label]

    for inj_id, results_by_fti in results_by_inj.items():
        plot_dir = os.path.join(
            get_mass_run_data_dir(mass_label),
            f"id_{inj_id}",
            "plots",
        )
        os.makedirs(plot_dir, exist_ok=True)

        for fti_param, result in results_by_fti.items():
            list_params = result["list_params"]
            positions = np.arange(1, len(list_params) + 1)
            y_values = [result["cv_bias_sigma"][p] for p in list_params]

            fig, ax = plt.subplots(figsize=(15, 6))
            ax.scatter(
                positions,
                y_values,
                color=plotutils.plotpalette[1],
                s=60,
            )
            ax.set_xticks(
                positions,
                labels=latex_param_labels(list_params),
                rotation=45,
                ha="right",
            )
            ax.set_ylabel(r"$\Delta\theta/\sigma$")
            ax.set_title(
                rf"CV bias, {mass_name}, inj_id={inj_id}, "
                rf"active FTI: {latex_param_label(fti_param)}"
            )
            ax.axhline(0.0, color="k", linewidth=1.0, alpha=0.5)
            for position in positions:
                ax.axvline(position, color="k", linewidth=0.3, alpha=0.3)

            plt.tight_layout()
            out_png = os.path.join(
                plot_dir,
                f"cv_bias_sigma_{mass_label}_id_{inj_id}_{fti_param}.png",
            )
            out_pdf = os.path.join(
                plot_dir,
                f"cv_bias_sigma_{mass_label}_id_{inj_id}_{fti_param}.pdf",
            )
            plt.savefig(out_png, dpi=200)
            plt.savefig(out_pdf)
            print("Saved:", out_png)
            print("Saved:", out_pdf)
            plt.show()


In [ ]:
# Summarize the active FTI biases for each mass and injection ID

for mass_label, results_by_inj in all_results.items():
    mass_name = MASS_TITLE_BY_LABEL[mass_label]

    for inj_id, results_by_fti in results_by_inj.items():
        fti_names = list(results_by_fti.keys())
        fti_bias = [
            results_by_fti[param]["cv_bias_sigma"][param]
            for param in fti_names
        ]
        positions = np.arange(1, len(fti_names) + 1)

        plot_dir = os.path.join(
            get_mass_run_data_dir(mass_label),
            f"id_{inj_id}",
            "plots",
        )
        os.makedirs(plot_dir, exist_ok=True)

        fig, ax = plt.subplots(figsize=(15, 6))
        ax.scatter(
            positions,
            fti_bias,
            color=plotutils.plotpalette[2],
            s=70,
        )
        ax.set_xticks(
            positions,
            labels=latex_param_labels(fti_names),
            rotation=45,
            ha="right",
        )
        ax.set_ylabel(r"Active FTI $\Delta\theta/\sigma$")
        ax.set_title(
            rf"Active-FTI bias summary, {mass_name}, inj_id={inj_id}"
        )
        ax.axhline(0.0, color="k", linewidth=1.0, alpha=0.5)
        for position in positions:
            ax.axvline(position, color="k", linewidth=0.3, alpha=0.3)

        plt.tight_layout()
        out_png = os.path.join(
            plot_dir,
            f"active_fti_bias_summary_{mass_label}_id_{inj_id}.png",
        )
        out_pdf = os.path.join(
            plot_dir,
            f"active_fti_bias_summary_{mass_label}_id_{inj_id}.pdf",
        )
        plt.savefig(out_png, dpi=200)
        plt.savefig(out_pdf)
        print("Saved:", out_png)
        print("Saved:", out_pdf)
        plt.show()


In [ ]:
# Compare all injection IDs for each mass and active FTI parameter
for mass_label, results_by_inj in all_results.items():
    mass_name = MASS_TITLE_BY_LABEL[mass_label]
    mass_output_dir = get_mass_run_data_dir(mass_label)

    for fti_param in FTI_PARAMS_TO_SCAN:
        fig, ax = plt.subplots(figsize=(15, 6))
        plot_params = list(BASE_GR_PARAM) + [fti_param]
        positions = np.arange(1, len(plot_params) + 1)

        available_ids = [
            inj_id for inj_id in sorted(results_by_inj)
            if fti_param in results_by_inj[inj_id]
        ]
        offsets = np.linspace(-0.24, 0.24, len(available_ids))

        for offset, inj_id in zip(offsets, available_ids):
            result = results_by_inj[inj_id][fti_param]
            y_values = [
                result["cv_bias_sigma"].get(param, np.nan)
                for param in plot_params
            ]
            label = (
                rf"inj_id={inj_id}, "
                + latex_domega_dtau_label(
                    result.get("d_omega"),
                    result.get("d_tau"),
                )
            )
            ax.scatter(
                positions + offset,
                y_values,
                label=label,
                s=60,
            )

        ax.set_xticks(
            positions,
            labels=latex_param_labels(plot_params),
            rotation=30,
            ha="right",
        )
        ax.set_ylabel(r"$\Delta\theta/\sigma$")
        ax.set_title(
            rf"CV bias, {mass_name}, FTI included: "
            rf"{latex_param_label(fti_param)}"
        )
        ax.axhline(0.0, color="k", linewidth=1.0, alpha=0.5)
        for position in positions:
            ax.axvline(position, color="k", linewidth=0.3, alpha=0.3)
        ax.legend(loc="best")
        plt.tight_layout()

        figure_name = (
            f"cv_bias_sigma_{mass_label}_all_inj_ids_"
            f"including_{fti_param}.png"
        )
        plt.savefig(os.path.join(mass_output_dir, figure_name), dpi=200)
        plt.show()


In [ ]:
# Comparing the active FTI biases across all injection IDs
for mass_label, results_by_inj in all_results.items():
    mass_name = MASS_TITLE_BY_LABEL[mass_label]
    plot_params = list(FTI_PARAMS_TO_SCAN)
    positions = np.arange(1, len(plot_params) + 1)

    fig, ax = plt.subplots(figsize=(18, 6))
    available_ids = sorted(results_by_inj.keys())
    offsets = np.linspace(-0.24, 0.24, len(available_ids))

    for offset, inj_id in zip(offsets, available_ids):
        y_values = []
        for fti_param in plot_params:
            if fti_param in results_by_inj[inj_id]:
                result = results_by_inj[inj_id][fti_param]
                y_values.append(
                    result["cv_bias_sigma"].get(fti_param, np.nan)
                )
            else:
                y_values.append(np.nan)

        sample_result = next(iter(results_by_inj[inj_id].values()))
        label = (
            rf"inj_id={inj_id}, "
            + latex_domega_dtau_label(
                sample_result.get("d_omega"),
                sample_result.get("d_tau"),
            )
        )
        ax.scatter(
            positions + offset,
            y_values,
            label=label,
            s=60,
        )

    ax.set_xticks(
        positions,
        labels=latex_param_labels(plot_params),
        rotation=45,
        ha="right",
    )
    ax.set_ylabel(r"$\Delta\theta_{\rm FTI}/\sigma$")
    ax.set_title(rf"Active FTI biases, {mass_name}")
    ax.axhline(0.0, color="k", linewidth=1.0, alpha=0.5)
    for position in positions:
        ax.axvline(position, color="k", linewidth=0.3, alpha=0.3)
    ax.legend(loc="best")
    plt.tight_layout()

    figure_name = f"cv_bias_sigma_all_FTI_params_{mass_label}.png"
    plt.savefig(
        os.path.join(get_mass_run_data_dir(mass_label), figure_name),
        dpi=200,
    )
    plt.show()


## 3. Frequency-domain waveform diagnostics

Here we compare the injected waveform, aligned GR recovery waveform, and residual for every mass, injection ID, and available TDI channel. The annotated panels also display the match and mismatch obtained after alignment.


In [ ]:
# Plotting the aligned waveforms and residuals for all masses
for mass_label, results_by_inj in alignment_results.items():
    mass_name = MASS_TITLE_BY_LABEL[mass_label]
    plot_dir = os.path.join(
        get_mass_run_data_dir(mass_label),
        "frequency_domain_plots",
    )
    os.makedirs(plot_dir, exist_ok=True)

    for inj_id in inj_ids:
        print("\n" + "#" * 80)
        print(f"Plotting {mass_label}, inj_id={inj_id}")
        print("#" * 80)

        align_result = results_by_inj[inj_id]
        tdidata = align_result["tdidata"]
        tdi_align = align_result["tdi_align"]
        residual_align = align_result["residual_align"]
        mismatch = align_result.get("mismatch")
        match = align_result.get("match")
        freq = np.asarray(tdidata["freq"])

        deviation_title = latex_domega_dtau_label(
            align_result.get("d_omega"),
            align_result.get("d_tau"),
        ) or f"inj_id={inj_id}"

        channels = [
            channel for channel in ("TDIA", "TDIE", "TDIT")
            if channel in tdidata and channel in tdi_align
        ]

        for channel in channels:
            h_inj = np.asarray(tdidata[channel])
            h_rec = np.asarray(tdi_align[channel])
            h_res = (
                np.asarray(residual_align[channel])
                if channel in residual_align
                else h_rec - h_inj
            )

            amp_inj = np.abs(h_inj)
            amp_rec = np.abs(h_rec)
            amp_res = np.abs(h_res)
            mask = (
                np.isfinite(freq)
                & np.isfinite(amp_inj)
                & np.isfinite(amp_rec)
                & np.isfinite(amp_res)
                & (freq > 0.0)
                & ((amp_inj > 0.0) | (amp_rec > 0.0) | (amp_res > 0.0))
            )

            if not np.any(mask):
                print(
                    f"Skipping {mass_label}, inj_id={inj_id}, {channel}: "
                    "no valid frequency points."
                )
                continue

            fig, ax = plt.subplots(figsize=(8, 5))
            ax.loglog(freq[mask], amp_inj[mask], label="Injection")
            ax.loglog(freq[mask], amp_rec[mask], "--", label="Recovery")
            ax.loglog(
                freq[mask],
                amp_res[mask],
                ":",
                label="Residual = Recovery - Injection",
            )
            ax.set_xlabel(r"$f\,[\mathrm{Hz}]$")
            ax.set_ylabel(r"$|\tilde{h}(f)|$")
            ax.set_title(
                rf"{mass_name}, {deviation_title}, "
                rf"TDI {latex_channel_label(channel)}",
                fontsize=10,
            )
            ax.grid(True, which="both", alpha=0.3)
            ax.legend(
                fontsize=7,
                frameon=True,
                markerscale=0.7,
                handlelength=1.2,
                handletextpad=0.3,
                borderpad=0.25,
                labelspacing=0.25,
                loc="best",
            )

            if mismatch is not None:
                match_text = (
                    rf"Match $={match:.6f}$" + "\n"
                    + rf"Mismatch $={mismatch:.3e}$"
                    if match is not None
                    else rf"Mismatch $={mismatch:.3e}$"
                )
                ax.text(
                    0.04,
                    0.06,
                    match_text,
                    transform=ax.transAxes,
                    fontsize=8,
                    verticalalignment="bottom",
                    horizontalalignment="left",
                    bbox={
                        "boxstyle": "round",
                        "facecolor": "white",
                        "alpha": 0.75,
                        "edgecolor": "gray",
                        "linewidth": 0.6,
                    },
                )

            plt.tight_layout()
            figure_file = os.path.join(
                plot_dir,
                f"freq_domain_{mass_label}_inj{inj_id}_{channel}.png",
            )
            plt.savefig(figure_file, dpi=200)
            plt.show()
            print("Saved:", figure_file)


In [ ]:
# Build one CV-bias collage for each mass
for mass_label, results_by_inj in all_results.items():
    mass_name = MASS_TITLE_BY_LABEL[mass_label]
    n_plots = len(FTI_PARAMS_TO_SCAN)
    ncols = 4
    nrows = math.ceil(n_plots / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(6.5 * ncols, 4.8 * nrows),
        squeeze=False,
    )
    global_handles = []
    global_labels = []

    for index, fti_param in enumerate(FTI_PARAMS_TO_SCAN):
        row = index // ncols
        column = index % ncols
        ax = axes[row, column]

        plot_params = list(BASE_GR_PARAM) + [fti_param]
        positions = np.arange(1, len(plot_params) + 1)
        available_ids = [
            inj_id for inj_id in sorted(results_by_inj)
            if fti_param in results_by_inj[inj_id]
        ]
        offsets = np.linspace(-0.24, 0.24, len(available_ids))

        for offset, inj_id in zip(offsets, available_ids):
            result = results_by_inj[inj_id][fti_param]
            y_values = [
                result["cv_bias_sigma"].get(param, np.nan)
                for param in plot_params
            ]
            label = (
                rf"inj_id={inj_id}, "
                + latex_domega_dtau_label(
                    result.get("d_omega"),
                    result.get("d_tau"),
                )
            )
            scatter = ax.scatter(
                positions + offset,
                y_values,
                label=label,
                s=45,
            )

            if index == 0:
                global_handles.append(scatter)
                global_labels.append(label)

        ax.set_xticks(positions)
        ax.set_xticklabels(
            latex_param_labels(plot_params),
            rotation=45,
            ha="right",
            fontsize=8,
        )
        ax.set_ylabel(r"$\Delta\theta/\sigma$", fontsize=10)
        ax.set_title(
            rf"FTI included: {latex_param_label(fti_param)}",
            fontsize=11,
        )
        ax.axhline(0.0, color="k", linewidth=1.0, alpha=0.5)
        for position in positions:
            ax.axvline(position, color="k", linewidth=0.3, alpha=0.3)
        ax.grid(alpha=0.25)

    for index in range(n_plots, nrows * ncols):
        row = index // ncols
        column = index % ncols
        axes[row, column].axis("off")

    fig.suptitle(
        rf"CV/systematic bias from {wf_name} recovery, {mass_name}",
        fontsize=16,
        y=0.995,
    )
    fig.legend(
        global_handles,
        global_labels,
        loc="lower center",
        ncol=min(len(global_labels), 4),
        fontsize=10,
        frameon=True,
    )
    plt.tight_layout(rect=[0, 0.06, 1, 0.96])

    figure_path = os.path.join(
        get_mass_run_data_dir(mass_label),
        f"cv_bias_sigma_collage_{mass_label}.png",
    )
    plt.savefig(figure_path, dpi=250, bbox_inches="tight")
    plt.show()
    print("Saved:", figure_path)
